# 🚄 SNCB/NMBS — Analyse Ponctualite Complete

**Objectif :** Dashboard complet de la ponctualite ferroviaire belge combinant donnees GTFS, API temps reel iRail, analyse statistique et cartographie.

**Stack :** Pandas, Plotly, Folium, iRail API

## 1. Imports & Configuration

In [92]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import requests
import zipfile
import io
import warnings
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'json'
import folium
from folium.plugins import MarkerCluster, HeatMap
from IPython.display import display, HTML

os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/clean', exist_ok=True)


import importlib
import src.map_generator
importlib.reload(src.map_generator)
from src.map_generator import MapGenerator

print("✅ Tous les imports sont fonctionnels")


✅ Tous les imports sont fonctionnels


## 2. Donnees GTFS Statiques (Horaires planifies)

In [93]:
GTFS_URL = 'https://gtfs.irail.be/nmbs/gtfs/latest.zip'

print(f'Telechargement GTFS depuis {GTFS_URL}...')
try:
    resp = requests.get(GTFS_URL, timeout=120)
    resp.raise_for_status()
    print(f'  ✅ {len(resp.content) / 1024 / 1024:.1f} Mo telecharges')
    
    zip_file = zipfile.ZipFile(io.BytesIO(resp.content))
    print(f'\n📁 Fichiers dans le ZIP :')
    for name in zip_file.namelist():
        if name.endswith('.txt'):
            info = zip_file.getinfo(name)
            print(f'   - {name} ({info.file_size / 1024:.1f} Ko)')
    
    dataframes = {}
    for name in zip_file.namelist():
        if name.endswith('.txt'):
            df_name = name.replace('.txt', '')
            dataframes[df_name] = pd.read_csv(zip_file.open(name))
            print(f'   ✅ {df_name} : {len(dataframes[df_name]):,} lignes x {len(dataframes[df_name].columns)} colonnes')
    
    print(f'\n📊 Total : {len(dataframes)} tables chargees')
except Exception as e:
    print(f'⚠️ Erreur GTFS : {e}')
    dataframes = {}

Telechargement GTFS depuis https://gtfs.irail.be/nmbs/gtfs/latest.zip...
  ✅ 13.8 Mo telecharges

📁 Fichiers dans le ZIP :
   - trips.txt (4193.2 Ko)
   - routes.txt (66.7 Ko)
   - translations.txt (107.3 Ko)
   - calendar.txt (811.1 Ko)
   - calendar_dates.txt (40691.2 Ko)
   - stop_time_overrides.txt (33995.0 Ko)
   - transfers.txt (85.1 Ko)
   - stop_times.txt (76083.3 Ko)
   - agency.txt (0.1 Ko)
   - feed_info.txt (0.2 Ko)
   - stops.txt (155.3 Ko)
   ✅ trips : 52,407 lignes x 9 colonnes
   ✅ routes : 1,359 lignes x 9 colonnes
   ✅ translations : 2,439 lignes x 7 colonnes
   ✅ calendar : 21,294 lignes x 10 colonnes
   ✅ calendar_dates : 2,314,877 lignes x 3 colonnes
   ✅ stop_time_overrides : 540,683 lignes x 4 colonnes
   ✅ transfers : 1,265 lignes x 8 colonnes
   ✅ stop_times : 981,622 lignes x 9 colonnes
   ✅ agency : 1 lignes x 6 colonnes
   ✅ feed_info : 1 lignes x 10 colonnes
   ✅ stops : 2,754 lignes x 12 colonnes

📊 Total : 11 tables chargees


### Exploration des gares (stops)

In [94]:
if 'stops' in dataframes:
    df_stops = dataframes['stops']
    print(f'Nombre de gares/arrets : {len(df_stops):,}')
    print(f'\nColonnes : {list(df_stops.columns)}')
    display(df_stops.head(5))
    
    print(f"\nGares sans coordonnees : {df_stops['stop_lat'].isna().sum()}")
else:
    print('Donnees stops non disponibles')

Nombre de gares/arrets : 2,754

Colonnes : ['stop_id', 'stop_code', 'stop_name', 'stop_desc', 'stop_lat', 'stop_lon', 'zone_id', 'stop_url', 'location_type', 'parent_station', 'platform_code', 'stop_timezone']


,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,platform_code,stop_timezone
0,8015190,NaN,Herzogenrath,NaN,50.870900,6.094400,NaN,NaN,0,NaN,NaN,NaN
1,8015199,NaN,Aix-la-Chapelle Ouest,NaN,50.780780,6.070550,NaN,NaN,0,NaN,NaN,NaN
2,8015321,NaN,KOLN-EHRENFELD,NaN,50.951538,6.914889,NaN,NaN,0,NaN,NaN,NaN
3,8015345,NaN,Aachen Hbf (DE),NaN,50.767760,6.091390,NaN,NaN,0,NaN,NaN,NaN
4,8015458,NaN,Köln Hbf (DE),NaN,50.943080,6.958970,NaN,NaN,0,NaN,NaN,NaN



Gares sans coordonnees : 0


### Exploration des lignes (routes)

In [95]:
if 'routes' in dataframes:
    df_routes = dataframes['routes']
    print(f'Nombre de lignes : {len(df_routes):,}')
    if 'route_short_name' in df_routes.columns:
        print(f'\nTop 10 types de service :')
        print(df_routes['route_short_name'].value_counts().head(10))
    display(df_routes.head(5))
else:
    print('Donnees routes non disponibles')

Nombre de lignes : 1,359

Top 10 types de service :
route_short_name
IC     437
L      235
BUS    219
P      151
EXT     30
T       26
TRN     22
S1      21
S61     19
S8      18
Name: count, dtype: int64


,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_url,route_color,route_text_color
0,1,NMBS/SNCB,L,Malines -- Muizen,NaN,100,NaN,NaN,NaN
1,10,NMBS/SNCB,L,Anvers-Central -- Gand-Saint-Pierre,NaN,100,NaN,NaN,NaN
2,100,NMBS/SNCB,L,Liège-Guillemins -- Glons,NaN,100,NaN,NaN,NaN
3,1000,NMBS/SNCB,BUS,Hasselt -- Kiewit,NaN,700,NaN,NaN,NaN
4,1001,NMBS/SNCB,BUS,Louvain -- Landen,NaN,700,NaN,NaN,NaN


## 3. API Temps Reel iRail

In [96]:
from src.realtime_api import RealtimeAPI

api = RealtimeAPI()
print('✅ API iRail initialisee')

✅ API iRail initialisee


### Chargement des gares depuis iRail

In [97]:
stations = api.get_stations()
print(f'Gares chargees depuis iRail : {len(stations)}')
if not stations.empty:
    display(stations.head(5))

Gares chargees depuis iRail : 713


,@id,id,name,locationX,locationY,standardname
0,http://irail.be/stations/NMBS/008400319,BE.NMBS.008400319,'s Hertogenbosch,5.294278,51.690420,'s Hertogenbosch
1,http://irail.be/stations/NMBS/008015345,BE.NMBS.008015345,Aachen Hbf,6.105275,50.770832,Aachen Hbf
2,http://irail.be/stations/NMBS/008015199,BE.NMBS.008015199,Aachen West,6.070550,50.780780,Aachen West
3,http://irail.be/stations/NMBS/008895000,BE.NMBS.008895000,Aalst,4.039653,50.942813,Aalst
4,http://irail.be/stations/NMBS/008895125,BE.NMBS.008895125,Aalst-Kerrebroek,4.024407,50.948377,Aalst-Kerrebroek


### Liveboard — Departs en temps reel

In [98]:
lb_gent = api.get_liveboard('Gent-Sint-Pieters', arrdep='departure')
print(f'Departs depuis Gent-Sint-Pieters : {len(lb_gent)}')
if not lb_gent.empty:
    display(lb_gent.head(10))

Departs depuis Gent-Sint-Pieters : 13


,station,vehicle,time,delay,canceled,left,platform,destination,destination_id,vehicle_shortname,scheduled_datetime,delay_min
0,Ghent-Sint-Pieters,BE.NMBS.L571,1775334060,0,0,0,11,Dendermonde,BE.NMBS.008893401,L 571,2026-04-04 20:21:00,0.0
1,Ghent-Sint-Pieters,BE.NMBS.IC2843,1775334120,0,0,0,12,Knokke,BE.NMBS.008891660,IC 2843,2026-04-04 20:22:00,0.0
2,Ghent-Sint-Pieters,BE.NMBS.IC521,1775334300,0,0,0,10,Welkenraedt,BE.NMBS.008844503,IC 521,2026-04-04 20:25:00,0.0
3,Ghent-Sint-Pieters,BE.NMBS.IC743,1775334420,0,0,0,4,Sint-Niklaas,BE.NMBS.008894508,IC 743,2026-04-04 20:27:00,0.0
4,Ghent-Sint-Pieters,BE.NMBS.IC721,1775335020,0,0,0,6,Poperinge,BE.NMBS.008896735,IC 721,2026-04-04 20:37:00,0.0
5,Ghent-Sint-Pieters,BE.NMBS.IC543,1775335020,840,0,0,12,Oostende,BE.NMBS.008891702,IC 543,2026-04-04 20:37:00,14.0
6,Ghent-Sint-Pieters,BE.NMBS.L593,1775335380,0,0,0,4,Brugge,BE.NMBS.008891009,L 593,2026-04-04 20:43:00,0.0
7,Ghent-Sint-Pieters,BE.NMBS.S53672,1775335380,0,0,0,5,Lokeren,BE.NMBS.008894201,S53 672,2026-04-04 20:43:00,0.0
8,Ghent-Sint-Pieters,BE.NMBS.IC1822,1775335920,0,0,0,4,Antwerp-Central,BE.NMBS.008821006,IC 1822,2026-04-04 20:52:00,0.0
9,Ghent-Sint-Pieters,BE.NMBS.S51793,1775336220,0,0,0,7,Oudenaarde,BE.NMBS.008892601,S51 793,2026-04-04 20:57:00,0.0


In [99]:
lb_brussels = api.get_liveboard('Brussels-Central', arrdep='departure')
print(f'Departs depuis Brussels-Central : {len(lb_brussels)}')
if not lb_brussels.empty:
    display(lb_brussels.head(10))

Departs depuis Brussels-Central : 33


,station,vehicle,time,delay,canceled,left,platform,destination,destination_id,vehicle_shortname,scheduled_datetime,delay_min
0,Brussels-Central,BE.NMBS.IC543,1775332560,1440,0,0,2,Oostende,BE.NMBS.008891702,IC 543,2026-04-04 19:56:00,24.0
1,Brussels-Central,BE.NMBS.IC2820,1775333880,0,0,0,1,Liège-Guillemins,BE.NMBS.008841004,IC 2820,2026-04-04 20:18:00,0.0
2,Brussels-Central,BE.NMBS.IC1543,1775334000,240,0,0,6,Blankenberge,BE.NMBS.008891405,IC 1543,2026-04-04 20:20:00,4.0
3,Brussels-Central,BE.NMBS.IC3222,1775334120,0,0,0,1,Dendermonde,BE.NMBS.008893401,IC 3222,2026-04-04 20:22:00,0.0
4,Brussels-Central,BE.NMBS.IC2142,1775334240,0,0,0,4,Brussels-South/Brussels-Midi,BE.NMBS.008814001,IC 2142,2026-04-04 20:24:00,0.0
5,Brussels-Central,BE.NMBS.IC2319,1775334480,0,0,0,3,Brussels Airport - Zaventem,BE.NMBS.008819406,IC 2319,2026-04-04 20:28:00,0.0
6,Brussels-Central,BE.NMBS.IC3343,1775334660,60,0,0,6,Brussels-South/Brussels-Midi,BE.NMBS.008814001,IC 3343,2026-04-04 20:31:00,1.0
7,Brussels-Central,BE.NMBS.IC3321,1775334720,0,0,0,5,Antwerp-Central,BE.NMBS.008821006,IC 3321,2026-04-04 20:32:00,0.0
8,Brussels-Central,BE.NMBS.IC2345,1775334720,120,0,0,4,Kortrijk,BE.NMBS.008896008,IC 2345,2026-04-04 20:32:00,2.0
9,Brussels-Central,BE.NMBS.IC2122,1775335020,0,0,0,3,Namur,BE.NMBS.008863008,IC 2122,2026-04-04 20:37:00,0.0


In [100]:
lb_liege = api.get_liveboard('Liege-Guillemins', arrdep='departure')
print(f'Departs depuis Liege-Guillemins : {len(lb_liege)}')
if not lb_liege.empty:
    display(lb_liege.head(10))

Departs depuis Liege-Guillemins : 10


,station,vehicle,time,delay,canceled,left,platform,destination,destination_id,vehicle_shortname,scheduled_datetime,delay_min
0,Liège-Guillemins,BE.NMBS.S415022,1775333940,0,0,0,2,Welkenraedt,BE.NMBS.008844503,S41 5022,2026-04-04 20:19:00,0.0
1,Liège-Guillemins,BE.NMBS.S415042,1775335500,0,0,0,3,Liège-Saint-Lambert,BE.NMBS.008841525,S41 5042,2026-04-04 20:45:00,0.0
2,Liège-Guillemins,BE.NMBS.IC13632,1775335500,0,0,0,9,Flemalle-Haute,BE.NMBS.008843208,IC 13632,2026-04-04 20:45:00,0.0
3,Liège-Guillemins,BE.NMBS.ICE318,1775335560,600,0,0,2,Brussels-South/Brussels-Midi,BE.NMBS.008814001,ICE 318,2026-04-04 20:46:00,10.0
4,Liège-Guillemins,BE.NMBS.L5593,1775335860,240,0,0,3,Liers,BE.NMBS.008841673,L 5593,2026-04-04 20:51:00,4.0
5,Liège-Guillemins,BE.NMBS.S425492,1775336220,0,0,0,7,Liers,BE.NMBS.008841673,S42 5492,2026-04-04 20:57:00,0.0
6,Liège-Guillemins,BE.NMBS.IC520,1775336580,0,0,0,1,Welkenraedt,BE.NMBS.008844503,IC 520,2026-04-04 21:03:00,0.0
7,Liège-Guillemins,BE.NMBS.IC545,1775336640,0,0,0,3,Oostende,BE.NMBS.008891702,IC 545,2026-04-04 21:04:00,0.0
8,Liège-Guillemins,BE.NMBS.S4318981,1775336880,0,0,0,2,Aachen Hbf,BE.NMBS.008015345,S43 18981,2026-04-04 21:08:00,0.0
9,Liège-Guillemins,BE.NMBS.IC13633,1775337420,0,0,0,9,Liers,BE.NMBS.008841673,IC 13633,2026-04-04 21:17:00,0.0


### Combinaison des donnees liveboard

In [101]:
all_liveboards = pd.concat([lb_gent, lb_brussels, lb_liege], ignore_index=True)
print(f'Total departes surveilles : {len(all_liveboards)}')

if not all_liveboards.empty:
    all_liveboards['on_time'] = all_liveboards['delay_min'] <= 5
    print(f"\nTrains a l'heure (<= 5 min) : {all_liveboards['on_time'].sum()}")
    print(f"Trains en retard (> 5 min) : {(~all_liveboards['on_time']).sum()}")
    print(f"Ponctualite : {all_liveboards['on_time'].mean() * 100:.1f}%")
    print(f"Retard moyen : {all_liveboards['delay_min'].mean():.1f} min")
    print(f"Retard max : {all_liveboards['delay_min'].max():.1f} min")

Total departes surveilles : 56

Trains a l'heure (<= 5 min) : 53
Trains en retard (> 5 min) : 3
Ponctualite : 94.6%
Retard moyen : 1.2 min
Retard max : 24.0 min


### Connections entre gares

In [102]:
conn = api.get_connections('Gent-Sint-Pieters', 'Brussels-Central')
print(f'Connections Gent -> Brussels : {len(conn)}')
if not conn.empty:
    display(conn)

Connections Gent -> Brussels : 6


,duration,departure_vehicle,departure_time,departure_delay,departure_canceled,departure_platform,departure_station,arrival_vehicle,arrival_time,arrival_delay,arrival_canceled,arrival_platform,arrival_station,duration_min,departure_delay_min,arrival_delay_min
0,2100,BE.NMBS.IC521,1775334300,0,0,10,Ghent-Sint-Pieters,BE.NMBS.IC521,1775336400,0,0,3,Brussels-Central,35.0,0.0,0.0
1,2160,BE.NMBS.IC522,1775339280,0,0,10,Ghent-Sint-Pieters,BE.NMBS.IC522,1775341440,0,0,1,Brussels-Central,36.0,0.0,0.0
2,2160,BE.NMBS.IC504,1775359440,0,0,10,Ghent-Sint-Pieters,BE.NMBS.IC504,1775361600,0,0,3,Brussels-Central,36.0,0.0,0.0
3,2220,BE.NMBS.IC1505,1775361780,0,0,10,Ghent-Sint-Pieters,BE.NMBS.IC1505,1775364000,0,0,3,Brussels-Central,37.0,0.0,0.0
4,2100,BE.NMBS.IC505,1775363100,0,0,10,Ghent-Sint-Pieters,BE.NMBS.IC505,1775365200,0,0,3,Brussels-Central,35.0,0.0,0.0
5,2220,BE.NMBS.IC1506,1775365380,0,0,11,Ghent-Sint-Pieters,BE.NMBS.IC1506,1775367600,0,0,3,Brussels-Central,37.0,0.0,0.0


In [103]:
conn2 = api.get_connections('Brussels-Central', 'Liege-Guillemins')
print(f'Connections Brussels -> Liege : {len(conn2)}')
if not conn2.empty:
    display(conn2)

Connections Brussels -> Liege : 6


,duration,departure_vehicle,departure_time,departure_delay,departure_canceled,departure_platform,departure_station,arrival_vehicle,arrival_time,arrival_delay,arrival_canceled,arrival_platform,arrival_station,duration_min,departure_delay_min,arrival_delay_min
0,4500,BE.NMBS.IC2820,1775333880,0,0,1,Brussels-Central,BE.NMBS.IC2820,1775338380,0,0,1,Liège-Guillemins,75.0,0.0,0.0
1,4740,BE.NMBS.IC521,1775336460,0,0,3,Brussels-Central,BE.NMBS.IC521,1775341200,0,0,1,Liège-Guillemins,79.0,0.0,0.0
2,4800,BE.NMBS.IC522,1775341500,0,0,1,Brussels-Central,BE.NMBS.IC522,1775346300,0,0,3,Liège-Guillemins,80.0,0.0,0.0
3,3540,BE.NMBS.IC504,1775361660,0,0,3,Brussels-Central,BE.NMBS.IC504,1775365200,0,0,1,Liège-Guillemins,59.0,0.0,0.0
4,2880,BE.NMBS.IC3206,1775362920,0,0,1,Brussels-Central,BE.NMBS.ICE11,1775365800,0,0,1,Liège-Guillemins,48.0,0.0,0.0
5,3480,BE.NMBS.IC505,1775365260,0,0,3,Brussels-Central,BE.NMBS.IC505,1775368740,0,0,2,Liège-Guillemins,58.0,0.0,0.0


### Perturbations actuelles

In [104]:
disturbances = api.get_disturbances()
print(f'Perturbations actuelles : {len(disturbances)}')
if not disturbances.empty:
    display(disturbances.head(10))

Perturbations actuelles : 44


,id,title,description,start,end,link
0,0,Saint-Nicolas / Sint-Niklaas - Courtrai / Kort...,"Contrary to the mentioned information, the IC ...",,,http://www.belgianrail.be/jp/nmbs-realtime/hel...
1,1,Lierre / Lier - Mol,"Every day, from 4 to 10/04 Infrabel is working...",,,http://www.belgianrail.be/jp/nmbs-realtime/hel...
2,2,Namur / Namen - Ciney,On Sunday 3/05 Infrabel is working on the trac...,,,http://www.belgianrail.be/jp/nmbs-realtime/hel...
3,3,Luttre,"Every day, from 16/03 to 26/04 Infrabel is wor...",,,https://www.belgiantrain.be/nl/news/works-mana...
4,4,Hal / Halle,"During the weekends, from 28/03 to 12/04 and o...",,,https://www.belgiantrain.be/nl/news/works-Hal
5,5,Arlon / Aarlen - Athus,"Every day, from 28/03 to 12/04 works will take...",,,https://www.belgiantrain.be/nl/news/works-rese...
6,6,Arlon / Aarlen - Athus / Luxembourg (LU),During the weekends of 18-19 and 25-26/04 Infr...,,,https://www.belgiantrain.be/nl/news/works-arlo...
7,7,Malines / Mechelen - Louvain / Leuven,"During the weekends, from 11 to 26/04 Infrabel...",,,https://www.belgiantrain.be/fr/support/faq/faq...
8,8,Gand-Saint-Pierre / Gent-Sint-Pieters - Ostend...,On Saturday 11/04 Infrabel is working on the t...,,,https://www.belgianrail.be/jp/download/brail_h...
9,9,Charleroi-Central - Nivelles / Nijvel,"Every day, from 18 to 20/04 Infrabel is workin...",,,http://www.belgianrail.be/jp/nmbs-realtime/hel...


## 4. Analyse Statistique de la Ponctualite

In [105]:
if not all_liveboards.empty:
    from src.kpi_calculator import KPICalculator
    
    trip_data = all_liveboards.rename(columns={
        'vehicle_shortname': 'route_id',
        'delay_min': 'arrival_delay'
    }).copy()
    trip_data['trip_id'] = trip_data['vehicle']
    
    kpi = KPICalculator(trip_data)
    kpis = kpi.calculate_all()
    print(kpi.generate_summary())

Ponctualite SNCB/NMBS
Trains surveilles: 53
A l'heure: 94.6%
Retard moyen: 1.2 min
Retard median: 0.0 min
Retard max: 24.0 min
Retards severes (>15min): 1



### Distribution des retards

In [106]:
if not all_liveboards.empty:
    fig = px.histogram(
        all_liveboards,
        x='delay_min',
        nbins=50,
        title='Distribution des retards (minutes)',
        labels={'delay_min': 'Retard (min)', 'count': 'Nombre de trains'},
        color_discrete_sequence=['#1f77b4']
    )
    fig.add_vline(x=0, line_dash='dash', line_color='green', annotation_text='A l\'heure')
    fig.add_vline(x=5, line_dash='dash', line_color='orange', annotation_text='Seuil ponctualite')
    fig.add_vline(x=15, line_dash='dash', line_color='red', annotation_text='Retard severe')
    fig.update_layout(showlegend=True)
    display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))


### Ponctualite par gare

In [107]:
if not all_liveboards.empty:
    station_stats = all_liveboards.groupby('station').agg(
        total_trains=('vehicle', 'count'),
        avg_delay=('delay_min', 'mean'),
        max_delay=('delay_min', 'max'),
        on_time_pct=('on_time', 'mean'),
    ).round(1)
    station_stats['on_time_pct'] = (station_stats['on_time_pct'] * 100).round(1)
    display(station_stats.sort_values('avg_delay', ascending=False))
    
    fig = px.bar(
        station_stats.reset_index(),
        x='station',
        y='avg_delay',
        title='Retard moyen par gare',
        labels={'station': 'Gare', 'avg_delay': 'Retard moyen (min)'},
        color='avg_delay',
        color_continuous_scale='RdYlGn_r'
    )
    display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))


,total_trains,avg_delay,max_delay,on_time_pct
station,,,,
Liège-Guillemins,10,1.4,10.0,90.0
Brussels-Central,33,1.2,24.0,100.0
Ghent-Sint-Pieters,13,1.1,14.0,90.0


### Trains les plus en retard

In [108]:
if not all_liveboards.empty:
    severe = all_liveboards[all_liveboards['delay_min'] > 15].sort_values('delay_min', ascending=False)
    print(f'Trains avec retard > 15 min : {len(severe)}')
    if len(severe) > 0:
        display(severe[['station', 'vehicle_shortname', 'destination', 'delay_min', 'scheduled_datetime']].head(20))

Trains avec retard > 15 min : 1


,station,vehicle_shortname,destination,delay_min,scheduled_datetime
13,Brussels-Central,IC 543,Oostende,24.0,2026-04-04 19:56:00


### Tableau de bord resume

In [109]:
if not all_liveboards.empty:
    summary_df = pd.DataFrame({
        'KPI': [
            'Total trains surveilles',
            'Trains a l\'heure (<= 5 min)',
            'Trains en retard (> 5 min)',
            'Ponctualite (%)',
            'Retard moyen (min)',
            'Retard median (min)',
            'Retard max (min)',
            'Retards severes (> 15 min)',
            'Gares surveillees',
            'Perturbations actuelles',
        ],
        'Valeur': [
            len(all_liveboards),
            int(all_liveboards['on_time'].sum()),
            int((~all_liveboards['on_time']).sum()),
            f"{all_liveboards['on_time'].mean() * 100:.1f}%",
            f"{all_liveboards['delay_min'].mean():.1f}",
            f"{all_liveboards['delay_min'].median():.1f}",
            f"{all_liveboards['delay_min'].max():.1f}",
            len(severe) if not all_liveboards.empty else 0,
            all_liveboards['station'].nunique(),
            len(disturbances),
        ]
    })
    display(summary_df)

,KPI,Valeur
0,Total trains surveilles,56
1,Trains a l'heure (<= 5 min),53
2,Trains en retard (> 5 min),3
3,Ponctualite (%),94.6%
4,Retard moyen (min),1.2
5,Retard median (min),0.0
6,Retard max (min),24.0
7,Retards severes (> 15 min),1
8,Gares surveillees,3
9,Perturbations actuelles,44


## 5. Cartographie Interactive

In [110]:
from src.map_generator import MapGenerator

map_gen = MapGenerator()
m = map_gen.create_base_map()
print('✅ Carte de base creee')

✅ Carte de base creee


### Gares sur la carte

In [111]:
if not stations.empty:
    stops_df = stations.rename(columns={'locationY': 'stop_lat', 'locationX': 'stop_lon', 'name': 'stop_name'})
    map_gen.add_stops(stops_df, max_stops=300)
    print(f'✅ {min(300, len(stops_df))} gares ajoutees a la carte')
    
    m = map_gen.get_map()
    display(HTML(m._repr_html_()))


✅ 300 gares ajoutees a la carte


### Positions des trains (donnees simulées pour visualisation)

In [112]:
np.random.seed(42)
n_trains = 50
train_positions = pd.DataFrame({
    'trip_id': [f'IC-{i}' for i in range(n_trains)],
    'route_id': [f'IC-{np.random.randint(1, 6)}' for _ in range(n_trains)],
    'latitude': np.random.uniform(49.5, 51.5, n_trains),
    'longitude': np.random.uniform(2.5, 6.5, n_trains),
    'departure_delay': np.random.exponential(5, n_trains),
    'arrival_delay': np.random.exponential(5, n_trains),
})

map_gen.add_train_positions(train_positions)
map_gen.add_legend()
print(f'✅ {n_trains} positions de trains ajoutees')

m = map_gen.get_map()
display(HTML(m._repr_html_()))


✅ 50 positions de trains ajoutees


### Heatmap de densite

In [113]:
map_heat = MapGenerator()
fig = map_heat.heatmap_plotly(train_positions)
if fig is not None:
    display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
    print("✅ Heatmap Plotly affichee")
else:
    print("❌ Aucune donnee pour la heatmap")


✅ Heatmap Plotly affichee


### Sauvegarde de la carte

In [114]:
map_gen.save('data/clean/sncb_map.html')
print('Carte sauvegardee dans data/clean/sncb_map.html')

Map saved to data/clean/sncb_map.html
Carte sauvegardee dans data/clean/sncb_map.html


## 6. Conclusions

In [115]:
print('=' * 60)
print('🚄 SNCB/NMBS — ANALYSE PONCTUALITE COMPLETE')
print('=' * 60)
print(f'\n📊 Donnees GTFS : {len(dataframes)} tables chargees')
if 'stops' in dataframes:
    print(f'   - {len(dataframes["stops"]):,} gares/arrets')
if 'routes' in dataframes:
    print(f'   - {len(dataframes["routes"]):,} lignes')
if 'trips' in dataframes:
    print(f'   - {len(dataframes["trips"]):,} trajets')

print(f'\n📡 API iRail :')
print(f'   - {len(stations)} gares')
print(f'   - {len(all_liveboards)} departes surveilles')
print(f'   - {len(disturbances)} perturbations actuelles')

if not all_liveboards.empty:
    print(f'\n📈 Ponctualite :')
    print(f'   - {all_liveboards["on_time"].mean() * 100:.1f}% des trains a l\'heure')
    print(f'   - Retard moyen : {all_liveboards["delay_min"].mean():.1f} min')
    print(f'   - Retard max : {all_liveboards["delay_min"].max():.1f} min')

print(f'\n🗺️ Cartes sauvegardees dans data/clean/')
print('\n✅ Analyse terminee avec succes!')

🚄 SNCB/NMBS — ANALYSE PONCTUALITE COMPLETE

📊 Donnees GTFS : 11 tables chargees
   - 2,754 gares/arrets
   - 1,359 lignes
   - 52,407 trajets

📡 API iRail :
   - 713 gares
   - 56 departes surveilles
   - 44 perturbations actuelles

📈 Ponctualite :
   - 94.6% des trains a l'heure
   - Retard moyen : 1.2 min
   - Retard max : 24.0 min

🗺️ Cartes sauvegardees dans data/clean/

✅ Analyse terminee avec succes!
